# COMP4680/8650 — Mini-Project 1: nanoGPT Story Generation

**Tasks covered:** Task 1 (Baseline) · Task 2 (Exploration) · Task 3 (Best Submission) · Task 4 (Arena, optional)

**Model constraint:** ≤ 32M parameters for Tasks 1–3 submission  
**Hardware:** Google Colab A100 (40 GB VRAM)

---
| Section | Description |
|---------|-------------|
| §0 | Environment setup (install, GPU check, repo) |
| §1 | **Task 1** — Official baseline (6L/6H/384D, ~30 M) |
| §2 | **Task 2** — Architecture ablations + instruction-tuning experiment |
| §3 | **Task 3** — Best ≤32 M checkpoint + HuggingFace upload |
| §4 | **Task 4** — Arena model (152 M, optional, no size limit) |

## §0 · Environment Setup

In [ ]:
# Install required packages (tiktoken for BPE tokenisation, datasets for HuggingFace)
!pip install -q tiktoken datasets huggingface_hub

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
    # TF32 gives free ~20% speedup on Ampere GPUs with negligible precision loss
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32  = True
print("\u2713 Dependencies ready")

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

# ── Edit these two variables to match your setup ─────────────────────────────
REPO_URL  = "https://github.com/Adithya-Rama/nano-llm.git"   # your GitHub repo
LOCAL_DIR = "/content/drive/MyDrive/COMP8650/Assgn-1/nano-llm/code"

if not os.path.exists(LOCAL_DIR):
    !git clone "{REPO_URL}" "{LOCAL_DIR}"
else:
    !cd "{LOCAL_DIR}" && git pull

os.chdir(LOCAL_DIR)
print(f"\u2713 Working directory: {os.getcwd()}")

---
## §1 · Task 1 — Baseline nanoGPT on ROCStories

### 1.1 Data Pipeline

**Dataset:** ROCStories (Mostafazadeh et al., 2016) — 78,528 five-sentence commonsense stories.
Downloaded from `mintujupally/ROCStories` on HuggingFace.

**Preprocessing steps:**
1. Stories are concatenated as plain text, separated by the GPT-2 `<|endoftext|>` token (id 50256).
2. Tokenised with `tiktoken` GPT-2 BPE (vocab = 50,257; padded to 50,304 for CUDA efficiency).
3. Split: 90% train (~3.7 M tokens) / 10% val (~410 K tokens) — fixed seed 42.
4. Written as raw `uint16` binary files (`train.bin`, `val.bin`) for fast memory-mapped loading during training.

> **No test stories are used during training.** The 19,633-story public test set is only used for final PPL evaluation.

In [ ]:
import os, numpy as np
os.chdir(LOCAL_DIR)

print("Preparing ROCStories (plain format for Task 1 + Task 3)...")
!python data/rocstories/prepare.py

# Verify output
for split in ['train', 'val']:
    p = f'data/rocstories/{split}.bin'
    n = len(np.fromfile(p, dtype=np.uint16))
    print(f"  {p}: {n:,} tokens")

### 1.2 Model Architecture

Following the professor's specification, the **Task 1 baseline** uses the official nanoGPT *baby GPT* configuration:

| Hyperparameter | Value |
|----------------|-------|
| `n_layer` | 6 |
| `n_head` | 6 |
| `n_embd` | 384 |
| `block_size` | 256 |
| `vocab_size` | 50,304 (GPT-2 BPE, padded) |
| **Total params** | **~30.2 M** (≤ 32 M ✓) |

Architecture is vanilla nanoGPT: learned positional embeddings, LayerNorm, GELU MLP — identical to the official `train_shakespeare_char.py` config (Karpathy, 2022).

**Reference:** Karpathy, A. (2022). nanoGPT. https://github.com/karpathy/nanoGPT

In [ ]:
import torch, sys
sys.path.insert(0, LOCAL_DIR)
from model import GPT, GPTConfig

cfg = GPTConfig(
    n_layer=6, n_head=6, n_embd=384, block_size=256,
    vocab_size=50304, bias=False, dropout=0.1,
    use_rmsnorm=False, use_rope=False,
    use_swiglu=False, use_qk_norm=False
)
model = GPT(cfg)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters : {n_params/1e6:.2f} M")
assert n_params <= 32e6, f"EXCEEDS 32M limit! ({n_params/1e6:.1f}M)"
print(f"Within 32M limit : \u2713")
del model; torch.cuda.empty_cache()

### 1.3 Training

**Key hyperparameters** (see `config/train_t1_baseline.py`):

| Hyperparameter | Value | Rationale |
|----------------|-------|-----------|
| `learning_rate` | 6e-4 | Standard peak LR for ~30 M GPT (Kaplan et al., 2020) |
| `max_iters` | 10,000 | ~88 passes through ROCStories |
| `batch_size` | 32 | Effective batch = 32 × 4 × 256 = 32,768 tokens/step |
| `warmup_iters` | 200 | Short warmup; small dataset |
| `label_smoothing` | 0.1 | Prevents overconfident predictions on small data |
| `dropout` | 0.1 | Regularisation |
| `dtype` | bfloat16 | Native A100 format; no quality loss |

LR schedule: cosine decay from 6e-4 → 6e-5.  
Checkpointing: every 15 minutes (time-based) + every eval_interval (250 steps) to survive Colab disconnects.

**Reference:** Kaplan, J. et al. (2020). Scaling laws for neural language models. arXiv:2001.08361.

In [ ]:
import os
os.chdir(LOCAL_DIR)

T1_CONFIG  = 'config/train_t1_baseline.py'
T1_OUT_DIR = 'out-t1-baseline'

ckpt = os.path.join(T1_OUT_DIR, 'ckpt.pt')
INIT_T1 = 'resume' if os.path.exists(ckpt) else 'scratch'
print(f"Checkpoint : {'found — resuming' if INIT_T1 == 'resume' else 'not found — training from scratch'}")
print(f"Config     : {T1_CONFIG}")
print(f"init_from  : {INIT_T1}")

In [ ]:
import os, subprocess
os.chdir(LOCAL_DIR)

# Safe to re-run: auto-resumes from checkpoint if one exists
result = subprocess.run(
    ['python', 'train.py', T1_CONFIG, f'--init_from={INIT_T1}'],
    check=False
)
print(f"\nTraining exit code: {result.returncode}")
print("Re-run this cell to resume if interrupted.")

### 1.4 Evaluation

**Perplexity (PPL)** is the primary quantitative metric:  
$$\text{PPL} = \exp\!\left(-\frac{1}{N}\sum_{i=1}^{N}\log P(w_i \mid w_{<i})\right)$$

Lower PPL = model assigns higher probability to the true next tokens.

**Passing threshold (prof announcement):** PPL ≤ 25.0 on the provided 19,633-story test set.

In [ ]:
import os, subprocess
os.chdir(LOCAL_DIR)

print("=" * 55)
print("TASK 1 — Perplexity on ROCStories test set")
print("=" * 55)
subprocess.run([
    'python', 'eval.py',
    '--init_from=resume',
    f'--out_dir={T1_OUT_DIR}',
    '--input_file=data/rocstories/eval_stories.txt'
], check=False)

print()
print("=" * 55)
print("TASK 1 — Qualitative story samples")
print("=" * 55)
subprocess.run([
    'python', 'sample_batch.py',
    '--init_from=resume',
    f'--out_dir={T1_OUT_DIR}',
    '--start=FILE:data/rocstories/eval_prompts.txt',
    '--batch_prompts=True',
    '--max_new_tokens=150',
    f'--output_file={T1_OUT_DIR}/generated_stories.jsonl'
], check=False)

---
## §2 · Task 2 — Exploration

### Research Questions

We investigate two complementary directions within the 32 M parameter budget:

**Direction 1 — Architecture ablation** (Cells 2.1–2.3):  
Which modern architectural components (RoPE, RMSNorm+SwiGLU, QK-Norm) individually
benefit story generation at the sub-32 M scale?

**Direction 2 — Mixed instruction training** (Cells 2.4–2.5):  
Can mixing instruction-prefixed stories into training improve narrative quality
without degrading perplexity on plain continuation prompts?
This is motivated by InstructGPT (Ouyang et al., 2022) but studied
here at a scale and domain (short narrative generation) that has not
been previously reported.

---
### 2.1 Dataset Preparation

Four variants are prepared:

| Dataset | Stories | Tokens | Purpose |
|---------|---------|--------|---------|
| `rocstories` | 78,528 | 3.7 M | Tasks 1 + ablations |
| `rocstories_structured` | 78,528 | 7.0 M | Structured-format ablation |
| `tinystories` | 500,000 | 107 M | Optional pretraining data |
| `mixed` | 78,528 × 3 formats | ~6.7 M | Instruction experiment |

In [ ]:
import os, numpy as np
os.chdir(LOCAL_DIR)

# 1. Structured format (already done if Task 1 ran — skip if bin exists)
if not os.path.exists('data/rocstories_structured/train.bin'):
    print("Preparing ROCStories structured format...")
    !python data/rocstories/prepare.py --structured
else:
    print("Structured dataset already exists — skipping")

# 2. TinyStories (may take ~5 min on first download)
if not os.path.exists('data/tinystories/train.bin'):
    print("\nPreparing TinyStories (~5 min first time)...")
    !python data/tinystories/prepare.py
else:
    print("TinyStories already exists — skipping")

# 3. Mixed instruction dataset
if not os.path.exists('data/mixed/train.bin'):
    print("\nBuilding mixed instruction dataset...")
    !python data/mixed/prepare.py
else:
    print("Mixed dataset already exists — skipping")

# Summary
print("\n" + "=" * 50)
print("Dataset Summary")
print("=" * 50)
for ds, split in [('rocstories','train'), ('rocstories','val'),
                   ('rocstories_structured','train'),
                   ('tinystories','train'), ('mixed','train'), ('mixed','val')]:
    p = f'data/{ds}/{split}.bin'
    if os.path.exists(p):
        n = len(np.fromfile(p, dtype=np.uint16))
        print(f"  {ds}/{split}.bin: {n/1e6:.2f}M tokens")

### 2.2 Architecture Ablation Design

We isolate **three orthogonal architectural modifications** from the vanilla nanoGPT:

| Config | Changes from vanilla | Out-dir |
|--------|---------------------|----------|
| A — Vanilla | none (baseline) | `out-t2-vanilla` |
| B — +RoPE | Rotary positional embeddings (Su et al., 2022) | `out-t2-rope` |
| C — +RMSNorm+SwiGLU | RMS layer norm + gated FFN (Zhang & Sennrich, 2019; Shazeer, 2020) | `out-t2-ffn` |
| D — +QK-Norm ★ | Query-Key normalisation (Henry et al., 2020; Gemma 2, 2024) | `out-t2-qknorm` |
| E — All Modern | All three combined | `out-t2-all-modern` |

★ **Novel contribution:** QK-Norm's effect on sub-32 M narrative-generation models is not previously reported.

All experiments use identical hyperparameters (6L/6H/384D, 10K steps, lr=6e-4) for a controlled comparison.
Each model is ~30.2 M parameters, well within the 32 M constraint.

In [ ]:
import os, subprocess, json, time
os.chdir(LOCAL_DIR)

ABLATIONS = [
    ('config/train_t2_ablation_a.py', 'out-t2-vanilla',    'A. Vanilla'),
    ('config/train_t2_ablation_b.py', 'out-t2-rope',       'B. +RoPE'),
    ('config/train_t2_ablation_c.py', 'out-t2-ffn',        'C. +RMSNorm+SwiGLU'),
    ('config/train_t2_ablation_d.py', 'out-t2-qknorm',     'D. +QK-Norm (novel)'),
    ('config/train_t2_ablation_e.py', 'out-t2-all-modern', 'E. All Modern'),
]

ppl_results = {}
start_all = time.time()

for cfg, out_dir, label in ABLATIONS:
    ckpt = os.path.join(out_dir, 'ckpt.pt')
    init = 'resume' if os.path.exists(ckpt) else 'scratch'
    t0 = time.time()
    print(f"\n{'─'*55}")
    print(f"  {label}  [init_from={init}]")
    print(f"{'─'*55}")

    # Train
    subprocess.run(['python', 'train.py', cfg, f'--init_from={init}'], check=False)

    # Eval PPL
    r = subprocess.run(
        ['python', 'eval.py', '--init_from=resume', f'--out_dir={out_dir}',
         '--input_file=data/rocstories/eval_stories.txt'],
        capture_output=True, text=True, check=False
    )
    for line in r.stdout.split('\n'):
        print(f'  {line}')
        if 'ppl' in line.lower() or 'perplexity' in line.lower():
            ppl_results[label] = line.strip()

    # Generate stories
    subprocess.run([
        'python', 'sample_batch.py',
        '--init_from=resume', f'--out_dir={out_dir}',
        '--start=FILE:data/rocstories/eval_prompts.txt',
        '--batch_prompts=True', '--max_new_tokens=150',
        f'--output_file={out_dir}/generated_stories.jsonl'
    ], check=False)

    elapsed = (time.time() - t0) / 60
    print(f"  Done in {elapsed:.1f} min")

print(f"\n{'='*55}")
print("  Ablation PPL Summary")
print(f"{'='*55}")
for lbl, val in ppl_results.items():
    print(f"  {lbl:28s}: {val}")
print(f"Total time: {(time.time()-start_all)/60:.1f} min")

### 2.3 Novel Experiment — Mixed Instruction + Continuation Training

**Motivation:** Modern instruction-tuned LLMs (InstructGPT, LLaMA-chat) demonstrate that training
on both instruction prompts and raw continuations enables a single model to serve multiple
interaction modes. For small story-generation models, we hypothesise that instruction exposure
can improve narrative coherence without degrading PPL on plain continuation.

**Training mixture** (`data/mixed/train.bin`):

| Format | Proportion | Example |
|--------|-----------|----------|
| A — Plain continuation | 55% | `The boy went to a video arcade...` |
| B — Instruction-prefixed | 30% | `Write a story about: the boy went to a video arcade.\n<story>` |
| C — Structured XML | 15% | `<story><s1>...</s1>...<s5>...</s5></story>` |

The **validation set is plain ROCStories only**, ensuring PPL comparisons are directly comparable
to Task 1 and the ablation baselines.

**Reference:** Ouyang, L. et al. (2022). Training language models to follow instructions with
human feedback. arXiv:2203.02155.

In [ ]:
import os, subprocess
os.chdir(LOCAL_DIR)

# Train instruction-aware model on mixed dataset
# Config: all-modern 30M + mixed data
# This is ALSO used as the Task 3 submission model
T3_CONFIG  = 'config/train_t3_best.py'
T3_OUT_DIR = 'out-t3-best'

ckpt = os.path.join(T3_OUT_DIR, 'ckpt.pt')
init_t3 = 'resume' if os.path.exists(ckpt) else 'scratch'
print(f"Task 3 model: init_from={init_t3}")

subprocess.run(
    ['python', 'train.py', T3_CONFIG, f'--init_from={init_t3}'],
    check=False
)
print("\nInstruction model training complete (or interrupted — re-run to resume).")

### 2.4 Analysis and Visualisation

The following cell generates:
1. **Learning-curve overlay** — val loss across all 5 architecture ablations
2. **Bar chart** — final val loss per experiment
3. **Story quality comparison** — lexical diversity and avg length per model

All outputs are saved as `.png` files for the report.

In [ ]:
import os, json, subprocess
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
os.chdir(LOCAL_DIR)

# ── Discover completed experiments ───────────────────────────────────────────
ALL_EXPS = [
    ('out-t1-baseline',   'Task 1 Baseline'),
    ('out-t2-vanilla',    'A. Vanilla'),
    ('out-t2-rope',       'B. +RoPE'),
    ('out-t2-ffn',        'C. +RMSNorm+SwiGLU'),
    ('out-t2-qknorm',     'D. +QK-Norm'),
    ('out-t2-all-modern', 'E. All Modern'),
    ('out-t3-best',       'F. +Mixed Instr.'),
]

available = [
    (d, l, os.path.join(d, 'train_log.jsonl'))
    for d, l in ALL_EXPS
    if os.path.exists(os.path.join(d, 'train_log.jsonl'))
]

print(f"Found {len(available)}/{len(ALL_EXPS)} completed experiments:")
for d, l, p in available:
    print(f"  \u2713 {l:28s} — {p}")
missing = [(d, l) for d, l in ALL_EXPS
           if not os.path.exists(os.path.join(d, 'train_log.jsonl'))]
for d, l in missing:
    print(f"  \u2717 {l:28s} — not yet run")

if not available:
    print("\nNo completed experiments yet. Run the training cells first.")
else:
    COLORS = ['#555555','#E53935','#1E88E5','#43A047','#FB8C00','#8E24AA','#00ACC1']

    # ── 1. Learning-curve overlay ─────────────────────────────────────────
    fig, ax = plt.subplots(figsize=(10, 5))
    for (d, label, log_path), color in zip(available, COLORS):
        steps, val_losses = [], []
        with open(log_path) as f:
            for line in f:
                entry = json.loads(line)
                if entry.get('val_loss') is not None:
                    steps.append(entry['step'])
                    val_losses.append(entry['val_loss'])
        if steps:
            ax.plot(steps, val_losses, label=label, color=color, linewidth=1.8)
    ax.set_xlabel('Training Step', fontsize=12)
    ax.set_ylabel('Validation Loss', fontsize=12)
    ax.set_title('Task 2: Validation Loss — Architecture Ablation', fontsize=14, fontweight='bold')
    ax.legend(fontsize=9, loc='upper right')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('ablation_curves.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\n\u2713 Saved: ablation_curves.png")

    # ── 2. Bar chart of final val loss ─────────────────────────────────────
    bar_labels, bar_vals = [], []
    for d, label, log_path in available:
        with open(log_path) as f:
            lines = [l.strip() for l in f if l.strip()]
        for line in reversed(lines):
            entry = json.loads(line)
            if entry.get('val_loss') is not None:
                bar_labels.append(label)
                bar_vals.append(entry['val_loss'])
                break

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(bar_labels, bar_vals, color=COLORS[:len(bar_vals)],
                  edgecolor='white', linewidth=1.5)
    for bar, val in zip(bars, bar_vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
    ax.set_ylabel('Final Validation Loss', fontsize=12)
    ax.set_title('Task 2: Final Validation Loss by Configuration', fontsize=14, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    plt.xticks(rotation=20, ha='right', fontsize=9)
    plt.tight_layout()
    plt.savefig('ablation_bar_chart.png', dpi=150, bbox_inches='tight')
    plt.close()
    print("\u2713 Saved: ablation_bar_chart.png")

    # ── 3. Summary table ──────────────────────────────────────────────────
    print(f"\n  {'Config':30s} {'Final Val Loss':>15s}")
    print(f"  {'─'*47}")
    for lbl, val in sorted(zip(bar_labels, bar_vals), key=lambda x: x[1]):
        print(f"  {lbl:30s} {val:15.4f}")

    # ── 4. Story quality (lexical diversity) ─────────────────────────────
    print(f"\n  {'Config':30s} {'Stories':>8s} {'Avg Len':>8s} {'Type/Token':>12s}")
    print(f"  {'─'*60}")
    for d, label, _ in available:
        stories_path = os.path.join(d, 'generated_stories.jsonl')
        if not os.path.exists(stories_path):
            continue
        texts = []
        with open(stories_path) as f:
            for line in f:
                try:
                    obj = json.loads(line)
                    texts.append(obj.get('generated', obj.get('text', '')))
                except json.JSONDecodeError:
                    pass
        if not texts:
            continue
        all_words = ' '.join(texts).lower().split()
        ttr = len(set(all_words)) / max(len(all_words), 1)
        avg_len = sum(len(t.split()) for t in texts) / len(texts)
        print(f"  {label:30s} {len(texts):8d} {avg_len:8.1f} {ttr:12.4f}")

print(f"\n  Download ablation_curves.png and ablation_bar_chart.png for your report.")

---
## §3 · Task 3 — Best Checkpoint Submission

The best submission model is the **all-modern 30 M + mixed instruction training** model already
trained in §2.3 (`out-t3-best/ckpt.pt`).  No new training is needed here.

### 3.1 Final Evaluation

Comprehensive evaluation on the full public test set.

In [ ]:
import os, subprocess
os.chdir(LOCAL_DIR)

print("=" * 55)
print("TASK 3 — Final PPL on test set")
print("=" * 55)
subprocess.run([
    'python', 'eval.py',
    '--init_from=resume',
    '--out_dir=out-t3-best',
    '--input_file=data/rocstories/eval_stories.txt'
], check=False)

print()
print("=" * 55)
print("TASK 3 — Story generation (plain prompts)")
print("=" * 55)
subprocess.run([
    'python', 'sample_batch.py',
    '--init_from=resume', '--out_dir=out-t3-best',
    '--start=FILE:data/rocstories/eval_prompts.txt',
    '--batch_prompts=True', '--max_new_tokens=200',
    '--output_file=out-t3-best/generated_stories.jsonl'
], check=False)

print()
print("=" * 55)
print("TASK 3 — Story generation (instruction prompts)")
print("=" * 55)
# Also test instruction-mode generation to verify dual capability
instr_prompts = [
    'Write a story about: a girl who lost her dog.\n',
    'Tell me a 5-sentence story involving a birthday surprise.\n',
    'Generate a brief narrative about learning to ride a bike.\n',
]
import json as _json
with open('/tmp/instr_prompts.txt', 'w') as f:
    f.write('\n---\n'.join(instr_prompts))
subprocess.run([
    'python', 'sample_batch.py',
    '--init_from=resume', '--out_dir=out-t3-best',
    '--start=FILE:/tmp/instr_prompts.txt',
    '--batch_prompts=True', '--max_new_tokens=200',
    '--output_file=out-t3-best/generated_instruction_stories.jsonl'
], check=False)

### 3.2 Package and Upload to HuggingFace

Files uploaded per assignment specification (`hf_load.py`):
- `ckpt.pt` — final checkpoint
- `model.py` — modified architecture (with RoPE, RMSNorm, SwiGLU, QK-Norm)
- `sample_params.json` — generation hyperparameters used for sampling
- `eval.py` + `configurator.py` — for evaluator to load and run the model

**Submission repo:** `<HF_USERNAME>/nanoGPT_hw`

In [ ]:
import os, shutil, json, subprocess
from google.colab import userdata
os.chdir(LOCAL_DIR)

HF_USERNAME = "YOUR_HF_USERNAME"   # ← REPLACE
HF_TOKEN    = userdata.get('HF_TOKEN')   # store token in Colab Secrets
os.environ['HF_TOKEN'] = HF_TOKEN

BEST_OUT = 'out-t3-best'
SUBMIT   = 'submission_hf'
os.makedirs(SUBMIT, exist_ok=True)

# Copy required files
for src, dst in [
    (f'{BEST_OUT}/ckpt.pt',  f'{SUBMIT}/ckpt.pt'),
    ('model.py',             f'{SUBMIT}/model.py'),
    ('sample_params.json',   f'{SUBMIT}/sample_params.json'),
    ('eval.py',              f'{SUBMIT}/eval.py'),
    ('configurator.py',      f'{SUBMIT}/configurator.py'),
]:
    if os.path.exists(src):
        shutil.copy(src, dst)

print("Submission folder contents:")
for fname in sorted(os.listdir(SUBMIT)):
    mb = os.path.getsize(f'{SUBMIT}/{fname}') / 1e6
    print(f"  {fname:<30s} {mb:.1f} MB")

REPO_ID = f"{HF_USERNAME}/nanoGPT_hw"
result = subprocess.run([
    'python', 'hf_load.py', 'upload',
    '--local-dir', SUBMIT,
    '--repo-id',   REPO_ID,
    '--token',     HF_TOKEN,
    '--commit-message', 'Final nanoGPT submission (~30M, all-modern + mixed instruction)'
], check=False)

print(f"\n\u2713 Uploaded to: https://huggingface.co/{REPO_ID}")
print(f"  Canvas submission:  {REPO_ID}")
print(f"  HF token:           {HF_TOKEN[:8]}...  (keep this secret!)")

---
## §4 · Task 4 — Arena Competition (Optional, 152 M)

> **Note:** This section is entirely optional and only relevant for the **arena competition**.
> The 152 M model **must NOT be submitted** to HuggingFace for Task 3 grading — it exceeds the 32 M constraint.

The arena model uses the same LLaMA-style architecture scaled to 152 M (12L/12H/768D),
trained on the combined ROCStories + TinyStories dataset (~110 M tokens) for richer language capacity.

### Generation strategy for human judging:
- Temperature: 0.85 (slightly higher for more creative, varied stories)
- Top-K: 50, Top-P: 0.9
- Repetition penalty: 1.2 (prevents looping, critical for coherence)

In [ ]:
import os, numpy as np
os.chdir(LOCAL_DIR)

# Prepare combined dataset (ROCStories + TinyStories)
if not os.path.exists('data/combined/train.bin'):
    print("Building combined dataset...")
    !python data/combined/prepare.py
else:
    arr = np.fromfile('data/combined/train.bin', dtype=np.uint16)
    print(f"Combined dataset already exists: {len(arr)/1e6:.0f}M tokens")

In [ ]:
import os, subprocess
os.chdir(LOCAL_DIR)

T4_CONFIG  = 'config/train_t4_arena.py'
T4_OUT_DIR = 'out-t4-arena'

ckpt = os.path.join(T4_OUT_DIR, 'ckpt.pt')
init_t4 = 'resume' if os.path.exists(ckpt) else 'scratch'
print(f"Arena model (152M): init_from={init_t4}")
print("Estimated training time: ~40-50 min on A100")

subprocess.run(
    ['python', 'train.py', T4_CONFIG, f'--init_from={init_t4}'],
    check=False
)
print("\nArena model ready. Re-run to resume if interrupted.")

In [ ]:
# Run this cell in PARALLEL with any long training cell to prevent Colab idle disconnect
import time, threading
from IPython.display import display, Javascript

def keep_alive():
    while True:
        time.sleep(55)
        try:
            display(Javascript('console.log("keep-alive")'))
        except Exception:
            pass
        print(f"  [keep-alive] {time.strftime('%H:%M:%S')}", end="\r")

t = threading.Thread(target=keep_alive, daemon=True)
t.start()
print("\u2713 Keep-alive thread started (pings every 55 s)")